In [1]:
import numpy as np
import cv2

# Simulated intrinsic matrix (typical dashcam/bus camera, 1280x720)
K = np.array([
    [900,   0, 640],
    [  0, 900, 360],
    [  0,   0,   1]
], dtype=np.float64)

dist_coeffs = np.zeros(5)  # assume mostly undistorted for simulation

In [3]:
pip install filterpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=b5637bab4ef66d2090e98a248e6d4e3ea61616d8b9a0667ad18fa962f0d0c13a
  Stored in directory: /root/.cache/pip/wheels/79/33/43/53b597b8f63de80842202a5fed633eea6f5ce3e3f6c6efbab8
Successfully built filterpy


In [4]:
from filterpy.kalman import ExtendedKalmanFilter as EKF
import numpy as np

class VehiclePoseEKF:
    def __init__(self):
        self.ekf = EKF(dim_x=5, dim_z=2)  # state: [lat, lon, heading, v, yaw_rate]
        self.ekf.x = np.array([22.7196, 75.8577, 0, 0, 0])  # init at Indore coords
        self.ekf.P *= 0.1
        self.ekf.R = np.diag([1e-6, 1e-6])  # GPS noise
        self.ekf.Q = np.eye(5) * 1e-4       # process noise

    def predict(self, dt, yaw_rate, velocity):
        theta = self.ekf.x[2]
        self.ekf.x[0] += velocity * np.cos(theta) * dt * 9e-6  # deg/meter approx
        self.ekf.x[1] += velocity * np.sin(theta) * dt * 9e-6
        self.ekf.x[2] += yaw_rate * dt

    def update_gps(self, lat, lon):
        z = np.array([lat, lon])
        self.ekf.update(z, HJacobian=lambda x: np.eye(2, 5), Hx=lambda x: x[:2])

In [6]:
pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 1.4 MB/s eta 0:00:00


In [7]:
import osmnx as ox

G = ox.graph_from_point((22.7196, 75.8577), dist=2000, network_type='drive')
route_nodes = list(G.nodes)[:50]
route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in route_nodes]
# Interpolate + add GPS noise + fake IMU yaw-rate to simulate a bus driving this route

In [8]:
def pixel_to_ground(u, v, K, camera_height, pitch_angle):
    fx, fy = K[0,0], K[1,1]
    cx, cy = K[0,2], K[1,2]

    # Ray direction in camera frame
    x = (u - cx) / fx
    y = (v - cy) / fy
    ray = np.array([x, y, 1.0])

    # Rotate ray by pitch to align with ground plane
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(pitch_angle), -np.sin(pitch_angle)],
        [0, np.sin(pitch_angle),  np.cos(pitch_angle)]
    ])
    ray_world = Rx @ ray

    # Intersect with ground plane (y = -camera_height)
    if ray_world[1] == 0:
        return None
    scale = -camera_height / ray_world[1]
    ground_point = ray_world * scale  # (X_forward, 0, Z_lateral) in meters relative to camera

    return ground_point[0], ground_point[2]  # (forward_m, lateral_m)

In [9]:
from geopy.distance import distance
from geopy import Point

def offset_to_latlon(lat, lon, heading_deg, forward_m, lateral_m):
    # Move forward along heading
    p1 = distance(meters=forward_m).destination(Point(lat, lon), bearing=heading_deg)
    # Move laterally (perpendicular to heading)
    p2 = distance(meters=lateral_m).destination(p1, bearing=(heading_deg + 90) % 360)
    return p2.latitude, p2.longitude

In [15]:
import numpy as np

# --- Step 1: Intrinsics ---
K = np.array([
    [900,   0, 640],
    [  0, 900, 360],
    [  0,   0,   1]
], dtype=np.float64)
dist_coeffs = np.zeros(5)

# --- Step 2: Extrinsics ---
camera_height = 2.2
pitch_angle = np.radians(10)
yaw_offset = 0.0

# --- bus pose (for demo, hardcode a sample point) ---
bus_lat, bus_lon = 22.7196, 75.8577
bus_heading = 45.0  # degrees

# --- Step 4: pixel -> ground ---
def pixel_to_ground(u, v, K, camera_height, pitch_angle):
    fx, fy = K[0,0], K[1,1]
    cx, cy = K[0,2], K[1,2]

    x = (u - cx) / fx
    y = (v - cy) / fy
    ray = np.array([x, y, 1.0])

    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(pitch_angle), -np.sin(pitch_angle)],
        [0, np.sin(pitch_angle),  np.cos(pitch_angle)]
    ])
    ray_world = Rx @ ray

    if ray_world[1] == 0:
        return None
    scale = -camera_height / ray_world[1]
    ground_point = ray_world * scale
    return ground_point[0], ground_point[2]

# --- Step 5: ground offset -> lat/lon ---
from geopy.distance import distance
from geopy import Point

def offset_to_latlon(lat, lon, heading_deg, forward_m, lateral_m):
    p1 = distance(meters=forward_m).destination(Point(lat, lon), bearing=heading_deg)
    p2 = distance(meters=lateral_m).destination(p1, bearing=(heading_deg + 90) % 360)
    return p2.latitude, p2.longitude

In [16]:
u, v = 640, 620  # pixel of detected pothole (bottom of bbox)
forward_m, lateral_m = pixel_to_ground(u, v, K, camera_height, pitch_angle)
event_lat, event_lon = offset_to_latlon(bus_lat, bus_lon, bus_heading, forward_m, lateral_m)

In [17]:
import json, time, random

def simulate_detection_stream(route_coords, n_events=20):
    events = []
    for i in range(n_events):
        idx = random.randint(0, len(route_coords)-2)
        lat, lon = route_coords[idx]
        # simulate a fake pixel detection + noisy GPS
        u, v = random.randint(400, 900), random.randint(500, 700)
        heading = random.uniform(0, 360)
        forward_m, lateral_m = pixel_to_ground(u, v, K, camera_height, pitch_angle)
        ev_lat, ev_lon = offset_to_latlon(lat, lon, heading, forward_m, lateral_m)
        events.append({
            "event_id": i,
            "class": random.choice(["pothole", "waterlogging", "missing_zebra_crossing"]),
            "confidence": round(random.uniform(0.7, 0.98), 2),
            "timestamp": time.time() + i,
            "lat": ev_lat, "lon": ev_lon,
            "bus_lat": lat, "bus_lon": lon, "heading": heading
        })
    return events

with open("simulated_events.json", "w") as f:
    json.dump(simulate_detection_stream(route_coords), f, indent=2)

In [19]:
import http.server
import socketserver
import webbrowser
import threading

PORT = 8000

def serve():
    handler = http.server.SimpleHTTPRequestHandler
    with socketserver.TCPServer(("", PORT), handler) as httpd:
        httpd.serve_forever()

threading.Thread(target=serve, daemon=True).start()
webbrowser.open(f"http://localhost:{PORT}/demo_map.html")

False

In [21]:
m = folium.Map(location=[22.7196, 75.8577], zoom_start=14, tiles="cartodbpositron")

In [23]:
import json

with open("simulated_events.json") as f:
    events = json.load(f)

In [24]:
import folium

m = folium.Map(
    location=[22.7196, 75.8577],
    zoom_start=14,
    tiles="cartodbpositron"
)

for ev in events:
    folium.CircleMarker(
        location=[ev["lat"], ev["lon"]],
        radius=5,
        popup=f"{ev['class']} ({ev['confidence']})",
        color="red" if ev["class"] == "pothole" else "blue"
    ).add_to(m)

m.save("demo_map.html")